In [ ]:
import json
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

sns.set_theme("talk", "ticks")

# Total MI varying kind of data

## MRI

In [ ]:
import glob
import os
import numpy as np
import matplotlib.pyplot as plt
import json
import re
from matplotlib.patches import Patch

listheo = []
listrue = []
statsReg = {}
for folder in glob.glob("../../NonLinearityResultsNew/eso245_cra_strin_*"):

    if os.path.isfile(
        os.path.join(folder, os.path.split(folder)[1] + "_globalStats.json")
    ):
        statsMI = np.load(os.path.join(folder, f"subject00_8.npy"))
        pairs = statsMI.shape[0]
        regions = int(0.5 + np.sqrt(0.25 + 2 * pairs))
        theo = int(re.findall(r"_(\d+)", folder)[0])
        listheo.append(theo)
        listrue.append(regions)
        print(folder, theo, regions, statsMI.shape)
        with open(
            os.path.join(folder, os.path.split(folder)[1] + "_globalStats.json")
        ) as fp:
            statsReg[regions] = {k: np.array(v) for k, v in json.load(fp).items()}
with open(
    "../../NonLinearityResultsNew/100center_bin8/100center_bin8_globalStats.json"
) as fp:
    statsReg[1000] = {k: np.array(v) for k, v in json.load(fp).items()}
# del statsReg[586]
order = np.argsort(listheo)
plt.plot(np.array(listheo)[order], np.array(listrue)[order], "o--")
plt.plot(np.array(listheo)[order], np.array(listheo)[order], ":k", lw=0.5)
plt.xlabel("Designed number of regions")
plt.ylabel("Effective number of regions")
plt.show()

In [ ]:
x = sorted(statsReg.keys())
all = np.zeros((19, len(x)))
for pat in range(19):
    y = [
        (
            2
            * (statsReg[i]["totalMI"][pat] - statsReg[i]["gaussMI"][pat])
            / (statsReg[i]["totalMI"][pat] + statsReg[i]["gaussMI"][pat])
            if len(statsReg[i]["totalMI"]) > pat
            else np.NaN
        )
        for i in x
    ]
    plt.plot(x, y, "o--b", lw=1, alpha=0.5)  # , label=pat)
    all[pat, :] = y
my = np.nanmean(all, axis=0)
sy = np.nanstd(all, axis=0)
plt.plot([x[0], x[-1]], [0, 0], "-r", lw=1, label=0, zorder=1)
plt.fill_between(
    x,
    my - 2 * sy,
    my + 2 * sy,
    color="yellow",
    alpha=1,
    edgecolor=None,
    label=r"$\pm 2 \sigma$",
)
plt.fill_between(
    x,
    my - sy,
    my + sy,
    color="green",
    alpha=0.5,
    edgecolor=None,
    label=r"$\pm 1 \sigma$",
)
plt.plot(x, my, "--k", lw=2, label="mean")
plt.ylabel("relative difference in MI")
plt.xlabel("# of regions")
plt.xscale("log")
plt.title("First 19 subjects")
plt.legend()
plt.show()

In [ ]:
x = sorted(statsReg.keys())
STRETCH = 1
all = np.zeros((19, len(x)))
y = []
for i in x:
    y.append(
        ((statsReg[i]["totalMI"] - statsReg[i]["gaussMI"]) / (statsReg[i]["totalMI"]))
    )  # 2*+statsReg[i]["gaussMI"]
(p1,) = plt.plot([-0.5, len(x) * STRETCH - 1], [0, 0], "-r", lw=1, label=0, zorder=1)
p2 = plt.hlines(
    0.1, -0.5, len(x) * STRETCH - 1, "r", "dashed", lw=1, label="10%", zorder=1
)
plt.boxplot(
    y,
    whis=2,
    positions=np.arange(len(x)) * STRETCH,
    boxprops={"color": "darkorange"},
    whiskerprops={"color": "darkorange"},
    flierprops={"color": "darkorange"},
    medianprops={"lw": 1.6, "color": "darkorange"},
    capprops={"color": "darkorange"},
)
tkpos = np.arange(len(x)) * STRETCH + 0
tickNames = [str(n) for n in x]
tickNames[-1] = "Sing.\nVoxel"
plt.xticks(tkpos, tickNames, rotation=45, fontsize="small")
plt.ylabel("Relative non-linearity in MI")
plt.xlabel("# of regions")
plt.legend(
    [Patch(facecolor="white", edgecolor="darkorange"), p1, p2],
    ["Total MI vs Gauss MI", r"0", r"10%"],
)
plt.title("245 subjects $-$ rs-fMRI")
plt.savefig("fMRI.pdf", bbox_inches="tight")
plt.show()

In [ ]:
x = sorted(statsReg.keys())
STRETCH = 1.5
all = np.zeros((19, len(x)))
y = []
for i in x:
    y.append(
        (
            2
            * (statsReg[i]["totalMI"] - statsReg[i]["gaussMI"])
            / (statsReg[i]["totalMI"] + statsReg[i]["gaussMI"])
        )
    )
(p1,) = plt.plot([-0.5, len(x) * STRETCH - 1], [0, 0], "-r", lw=1, label=0, zorder=1)
p2 = plt.hlines(
    0.1, -0.5, len(x) * STRETCH - 1, "r", "dashed", lw=1, label="10%", zorder=1
)
plt.boxplot(
    y,
    whis=2,
    positions=np.arange(len(x)) * STRETCH - 0.3,
    boxprops={"color": "darkorange"},
    whiskerprops={"color": "darkorange"},
    flierprops={"color": "darkorange"},
    medianprops={"lw": 1.6, "color": "darkorange"},
    capprops={"color": "darkorange"},
)
y = []
for i in x:
    y.append(
        (
            2
            * (statsReg[i]["totalMIshadow"] - statsReg[i]["gaussMIshadow"])
            / (statsReg[i]["totalMIshadow"] + statsReg[i]["gaussMIshadow"])
        )
    )
plt.boxplot(
    y,
    whis=2,
    positions=np.arange(len(x)) * STRETCH + 0.3,
    boxprops={"color": "blue"},
    whiskerprops={"color": "blue"},
    flierprops={"color": "blue"},
    medianprops={"lw": 1.6, "color": "blue"},
    capprops={"color": "blue"},
)
tkpos = np.arange(len(x)) * STRETCH + 0
tickNames = [str(n) for n in x]
tickNames[-1] = "Sing.\nVoxel"
plt.xticks(tkpos, tickNames, rotation=45, fontsize="small")
plt.ylabel("Relative non-linearity in MI")
plt.xlabel("# of regions")
plt.legend(
    [
        Patch(facecolor="white", edgecolor="darkorange"),
        Patch(facecolor="white", edgecolor="blue"),
        p1,
        p2,
    ],
    ["Total MI vs Gauss MI", "Shadow dataset", r"0%", r"10%"],
)
plt.title("245 subjects $-$ rs-fMRI")
plt.savefig("fMRISha.pdf", bbox_inches="tight")
plt.show()

In [ ]:
STRETCH = 2
fig = plt.figure(figsize=(12, 6))
bpx = []
for i in x:
    tmp = statsReg[i]["totalMI"]
    bpx.append([a for a in tmp if a != max(tmp)])
xpos = np.arange(len(x)) * STRETCH + 0.15
plt.boxplot(
    bpx,
    whis=2,
    positions=xpos,
    boxprops={"color": "b"},
    whiskerprops={"color": "b"},
    flierprops={"color": "b"},
    medianprops={"lw": 1.6, "color": "b"},
    capprops={"color": "b"},
)
bpx = []
for i in x:
    tmp = statsReg[i]["gaussMI"]
    bpx.append([a for a in tmp if a != max(tmp)])
xpos = np.arange(len(x)) * STRETCH + 0.85
plt.boxplot(
    bpx,
    whis=2,
    positions=xpos,
    boxprops={"color": "darkorange"},
    whiskerprops={"color": "darkorange"},
    flierprops={"color": "darkorange"},
    medianprops={"lw": 1.6, "color": "darkorange"},
    capprops={"color": "darkorange"},
)
tkpos = np.arange(len(x)) * STRETCH + 0.5
labels = x[:]
labels[-1] = "Sing.\nVoxel"
plt.xticks(tkpos, labels, rotation=45, fontsize="small")
plt.ylim(bottom=0)
cont = 0.15 * np.power(x, -1 / 4)
(p1,) = plt.plot(tkpos, cont, ":k", lw=1)
(p2,) = plt.plot(tkpos, cont, "xr", lw=1)
plt.legend(
    [
        Patch(facecolor="white", edgecolor="b"),
        Patch(facecolor="white", edgecolor="darkorange"),
        (p1, p2),
    ],
    ["Total MI", "Gauss MI", r"$\propto x^{-1/4}$"],
)
plt.xlabel("# of regions")
plt.ylabel("MI")
plt.show()

## EEG BLP electrode

In [ ]:
import glob
import os
import numpy as np
import matplotlib.pyplot as plt
import json
import re
from matplotlib.patches import Patch

statsReg = {}
for folder in glob.glob("../../NonLinearityResultsNew/FIX_electrode_BLP_band*"):
    if "NG" in folder:
        continue
    if os.path.isfile(
        os.path.join(folder, os.path.split(folder)[1] + "_globalStats.json")
    ):
        statsMI = np.load(os.path.join(folder, f"subject00_8.npy"))
        pairs = statsMI.shape[0]
        band = int(re.findall(r"band(\d+)", folder)[0])
        print(folder, band, statsMI.shape)
        with open(
            os.path.join(folder, os.path.split(folder)[1] + "_globalStats.json")
        ) as fp:
            statsReg[band] = {k: np.array(v) for k, v in json.load(fp).items()}

In [ ]:
IS_SHADOW = "shadow"  # ""#

In [ ]:
x = sorted(statsReg.keys())

all = np.zeros((19, len(x)))
for pat in range(19):
    y = [
        (
            2
            * (statsReg[i]["totalMI"][pat] - statsReg[i]["gaussMI"][pat])
            / (statsReg[i]["totalMI"][pat] + statsReg[i]["gaussMI"][pat])
            if len(statsReg[i]["totalMI"]) > pat
            else np.NaN
        )
        for i in x
    ]
    plt.plot(x, y, "o--b", lw=1, alpha=0.5)
    all[pat, :] = y
my = np.nanmean(all, axis=0)
sy = np.nanstd(all, axis=0)
plt.plot([x[0], x[-1]], [0, 0], "-r", lw=1, label=0, zorder=1)
plt.fill_between(
    x,
    my - 2 * sy,
    my + 2 * sy,
    color="yellow",
    alpha=1,
    edgecolor=None,
    label=r"$\pm 2 \sigma$",
)
plt.fill_between(
    x,
    my - sy,
    my + sy,
    color="green",
    alpha=0.5,
    edgecolor=None,
    label=r"$\pm 1 \sigma$",
)
plt.plot(x, my, "--k", lw=2, label="mean")
plt.xticks(
    ticks=np.arange(1, 10),
    labels=[
        r"$\delta$",
        r"$\theta$",
        r"$\alpha$",
        r"$\beta_{LOW}$",
        r"$\beta_{MID}$",
        r"$\beta_{HIGH}$",
        r"$\gamma$",
        r"$\beta_{ALL}$",
        r"$1-40 Hz$",
    ],
)
plt.ylabel("relative difference in MI")
plt.xlabel("Band")
plt.legend()
plt.title("First 19 subjects")
plt.show()

In [ ]:
x = sorted(statsReg.keys())

y = []
for i in x:
    y.append(
        (
            (statsReg[i]["totalMI" + IS_SHADOW] - statsReg[i]["gaussMI" + IS_SHADOW])
            / (statsReg[i]["totalMI" + IS_SHADOW])
        )
    )  # +statsReg[i]["gaussMI"+IS_SHADOW]
p1 = plt.hlines(0.0, 0.5, len(x) + 0.5, "r", "solid", lw=1, label="%", zorder=1)
p2 = plt.hlines(0.1, 0.5, len(x) + 0.5, "r", "dashed", lw=1, label="10%", zorder=1)
plt.boxplot(
    y,
    whis=2,
    boxprops={"color": "darkorange"},
    whiskerprops={"color": "darkorange"},
    flierprops={"color": "darkorange"},
    medianprops={"lw": 1.6, "color": "darkorange"},
    capprops={"color": "darkorange"},
)
tkpos = np.arange(len(x)) + 1
plt.xticks(tkpos, x, rotation=45, fontsize="small")
plt.ylabel("relative difference in MI")
plt.xlabel("Band")
plt.xticks(
    ticks=np.arange(1, 10),
    labels=[
        r"$\delta$",
        r"$\theta$",
        r"$\alpha$",
        r"$\beta_{LOW}$",
        r"$\beta_{MID}$",
        r"$\beta_{HIGH}$",
        r"$\gamma$",
        r"$\beta_{ALL}$",
        r"$1-40 Hz$",
    ],
)
plt.legend(
    [Patch(facecolor="white", edgecolor="darkorange"), p1, p2],
    ["Total MI vs Gauss MI", r"0", r"10%"],
)
plt.title("All subjects")
plt.show()
store_electrode = np.array(y)  # [[0,1,2,7,6]]

In [ ]:
fig = plt.figure(figsize=(12, 6))
bpx = []
for i in x:
    bpx.append(statsReg[i]["totalMI"])
xpos = 2 * np.array(x) - 0.8
plt.boxplot(
    bpx,
    whis=2,
    positions=xpos,
    boxprops={"color": "b"},
    whiskerprops={"color": "b"},
    flierprops={"color": "b"},
    medianprops={"lw": 1.6, "color": "b"},
    capprops={"color": "b"},
)
bpx = []
for i in x:
    bpx.append(statsReg[i]["gaussMI"])
xpos = 2 * np.array(x) - 0.2
plt.boxplot(
    bpx,
    whis=2,
    positions=xpos,
    boxprops={"color": "darkorange"},
    whiskerprops={"color": "darkorange"},
    flierprops={"color": "darkorange"},
    medianprops={"lw": 1.6, "color": "darkorange"},
    capprops={"color": "darkorange"},
)
tkpos = 2 * np.array(x) - 0.5
plt.xticks(
    tkpos,
    labels=[
        r"$\delta$",
        r"$\theta$",
        r"$\alpha$",
        r"$\beta_{LOW}$",
        r"$\beta_{MID}$",
        r"$\beta_{HIGH}$",
        r"$\gamma$",
        r"$\beta_{ALL}$",
        r"$1-40 Hz$",
    ],
    rotation=45,
    fontsize="small",
)
plt.ylim(bottom=0)
plt.legend(
    [
        Patch(facecolor="white", edgecolor="b"),
        Patch(facecolor="white", edgecolor="darkorange"),
    ],
    ["Total MI", "Gauss MI"],
)
plt.xlabel("Band")
plt.ylabel("MI")
plt.show()

## EEG BLP source

In [ ]:
import glob
import os
import numpy as np
import matplotlib.pyplot as plt
import json
import re
from matplotlib.patches import Patch

statsReg = {}
for folder in glob.glob("../../NonLinearityResultsNew/FIX_source_BLP_band*"):
    if os.path.isfile(
        os.path.join(folder, os.path.split(folder)[1] + "_globalStats.json")
    ):
        try:
            assert False
            statsMI = np.load(os.path.join(folder, f"subject00_8.npy"))
        except:
            statsMI = np.ndarray((0,))
        band = int(re.findall(r"band(\d+)", folder)[0])
        print(folder, band, statsMI.shape)
        with open(
            os.path.join(folder, os.path.split(folder)[1] + "_globalStats.json")
        ) as fp:
            statsReg[band] = {k: np.array(v) for k, v in json.load(fp).items()}

In [ ]:
x = sorted(statsReg.keys())

all = np.zeros((19, len(x)))
for pat in range(19):
    y = [
        (
            (statsReg[i]["totalMI"][pat] - statsReg[i]["gaussMI"][pat])
            / (statsReg[i]["totalMI"][pat])
            if len(statsReg[i]["totalMI"]) > pat
            else np.NaN
        )
        for i in x
    ]  # +2*statsReg[i]["gaussMI"][pat]
    plt.plot(x, y, "o--b", lw=1, alpha=0.5)
    all[pat, :] = y
my = np.nanmean(all, axis=0)
sy = np.nanstd(all, axis=0)
plt.plot([x[0], x[-1]], [0, 0], "-r", lw=1, label=0, zorder=1)
plt.fill_between(
    x,
    my - 2 * sy,
    my + 2 * sy,
    color="yellow",
    alpha=1,
    edgecolor=None,
    label=r"$\pm 2 \sigma$",
)
plt.fill_between(
    x,
    my - sy,
    my + sy,
    color="green",
    alpha=0.5,
    edgecolor=None,
    label=r"$\pm 1 \sigma$",
)
plt.plot(x, my, "--k", lw=2, label="mean")
plt.ylabel("relative difference in MI")
plt.xticks(
    ticks=np.arange(1, 10),
    labels=[
        r"$\delta$",
        r"$\theta$",
        r"$\alpha$",
        r"$\beta_{LOW}$",
        r"$\beta_{MID}$",
        r"$\beta_{HIGH}$",
        r"$\gamma$",
        r"$\beta_{ALL}$",
        r"$1-40 Hz$",
    ],
)
plt.xlabel("Band")
plt.legend()
plt.title("First 19 subjects")
plt.show()

In [ ]:
x = sorted(statsReg.keys())

y = []
for i in x:
    y.append(
        (
            (statsReg[i]["totalMI" + IS_SHADOW] - statsReg[i]["gaussMI" + IS_SHADOW])
            / (statsReg[i]["totalMI" + IS_SHADOW])
        )
    )  # 2*+statsReg[i]["gaussMI"+IS_SHADOW]
# p1,= plt.plot([1, len(x)], [0,0], "-r", lw=1, label=0, zorder=1)
p1 = plt.hlines(0.0, 0.5, len(x) + 0.5, "r", "solid", lw=1, label="0", zorder=1)
p2 = plt.hlines(0.1, 0.5, len(x) + 0.5, "r", "dashed", lw=1, label="10%", zorder=1)
plt.boxplot(
    y,
    whis=2,
    boxprops={"color": "darkorange"},
    whiskerprops={"color": "darkorange"},
    flierprops={"color": "darkorange"},
    medianprops={"lw": 1.6, "color": "darkorange"},
    capprops={"color": "darkorange"},
)
tkpos = np.arange(len(x)) + 1
plt.xticks(tkpos, x, rotation=45, fontsize="small")
plt.ylabel("relative difference in MI")
plt.xticks(
    ticks=np.arange(1, 10),
    labels=[
        r"$\delta$",
        r"$\theta$",
        r"$\alpha$",
        r"$\beta_{LOW}$",
        r"$\beta_{MID}$",
        r"$\beta_{HIGH}$",
        r"$\gamma$",
        r"$\beta_{ALL}$",
        r"$1-40 Hz$",
    ],
)
plt.xlabel("Band")
plt.legend(
    [Patch(facecolor="white", edgecolor="darkorange"), p1, p2],
    ["Total MI vs Gauss MI", r"0%", r"10%"],
)
plt.title("All subjects")
plt.show()
store_source = np.array(y)  # [[0,1,2,7,6]]

In [ ]:
fig = plt.figure(figsize=(12, 6))
bpx = []
for i in x:
    bpx.append(statsReg[i]["totalMI"])
xpos = 2 * np.array(x) - 0.8
plt.boxplot(
    bpx,
    whis=2,
    positions=xpos,
    boxprops={"color": "b"},
    whiskerprops={"color": "b"},
    flierprops={"color": "b"},
    medianprops={"lw": 1.6, "color": "b"},
    capprops={"color": "b"},
)
bpx = []
for i in x:
    bpx.append(statsReg[i]["gaussMI"])
xpos = 2 * np.array(x) - 0.2
plt.boxplot(
    bpx,
    whis=2,
    positions=xpos,
    boxprops={"color": "darkorange"},
    whiskerprops={"color": "darkorange"},
    flierprops={"color": "darkorange"},
    medianprops={"lw": 1.6, "color": "darkorange"},
    capprops={"color": "darkorange"},
)
tkpos = 2 * np.arange(1, 10) - 0.5
# tkpos = 2*np.array(x)-0.5
plt.xticks(
    tkpos,
    labels=[
        r"$\delta$",
        r"$\theta$",
        r"$\alpha$",
        r"$\beta_{LOW}$",
        r"$\beta_{MID}$",
        r"$\beta_{HIGH}$",
        r"$\gamma$",
        r"$\beta_{ALL}$",
        r"$1-40 Hz$",
    ],
    rotation=45,
    fontsize="small",
)
plt.ylim(bottom=0)
plt.legend(
    [
        Patch(facecolor="white", edgecolor="b"),
        Patch(facecolor="white", edgecolor="darkorange"),
    ],
    ["Total MI", "Gauss MI"],
)
plt.xlabel("Band")
plt.ylabel("MI")
plt.show()

## Raw electrode

In [ ]:
import glob
import os
import numpy as np
import matplotlib.pyplot as plt
import json
import re
from matplotlib.patches import Patch

statsReg = {}
for folder in glob.glob(
    "../../NonLinearityResultsNew/FIX_EEG_fixedSamples_band?_bin10"
):
    if os.path.isfile(
        os.path.join(folder, os.path.split(folder)[1] + "_globalStats.json")
    ):
        try:
            statsMI = np.load(os.path.join(folder, f"subject00_10.npy"))
        except:
            statsMI = np.array([])
        pairs = statsMI.shape[0]
        band = int(re.findall(r"band(\d+)", folder)[0])
        print(folder, band, statsMI.shape)
        with open(
            os.path.join(folder, os.path.split(folder)[1] + "_globalStats.json")
        ) as fp:
            statsReg[band] = {k: np.array(v) for k, v in json.load(fp).items()}

statsRegTime = {}
for folder in glob.glob("../NonLinearityResultsNew/FIX_EEG_fixedTime_band?_bin*"):
    if os.path.isfile(
        os.path.join(folder, os.path.split(folder)[1] + "_globalStats.json")
    ):
        try:
            assert False
            statsMI = np.load(os.path.join(folder, f"subject00_10.npy"))
        except:
            statsMI = np.array([])
        pairs = statsMI.shape[0]
        band = int(re.findall(r"band(\d+)", folder)[0])
        print(folder, band, statsMI.shape)
        with open(
            os.path.join(folder, os.path.split(folder)[1] + "_globalStats.json")
        ) as fp:
            statsRegTime[band] = {k: np.array(v) for k, v in json.load(fp).items()}

In [ ]:
x = sorted(statsReg.keys())

all = np.zeros((19, len(x)))
for pat in range(19):
    y = [
        (
            (statsReg[i]["totalMI"][pat] - statsReg[i]["gaussMI"][pat])
            / (statsReg[i]["totalMI"][pat])
            if len(statsReg[i]["totalMI"]) > pat
            else np.NaN
        )
        for i in x
    ]  # 2*+statsReg[i]["gaussMI"][pat]
    plt.plot(x, y, "o--b", lw=1, alpha=0.5)
    all[pat, :] = y
my = np.nanmean(all, axis=0)
sy = np.nanstd(all, axis=0)
plt.plot([x[0], x[-1]], [0, 0], "-r", lw=1, label=0, zorder=1)
plt.fill_between(
    x,
    my - 2 * sy,
    my + 2 * sy,
    color="yellow",
    alpha=1,
    edgecolor=None,
    label=r"$\pm 2 \sigma$",
)
plt.fill_between(
    x,
    my - sy,
    my + sy,
    color="green",
    alpha=0.5,
    edgecolor=None,
    label=r"$\pm 1 \sigma$",
)
plt.plot(x, my, "--k", lw=2, label="mean")
plt.xticks(
    ticks=np.arange(1, 10),
    labels=[
        r"$\delta$",
        r"$\theta$",
        r"$\alpha$",
        r"$\beta_{LOW}$",
        r"$\beta_{MID}$",
        r"$\beta_{HIGH}$",
        r"$\gamma$",
        r"$\beta_{ALL}$",
        r"$1-40 Hz$",
    ],
)
plt.ylabel("relative difference in MI")
plt.xlabel("Band")
plt.legend()
plt.title("First 19 subjects")
plt.show()

In [ ]:
STRETCH = 1.5
x = sorted(statsReg.keys())
basepos = np.arange(len(x)) * STRETCH
pos = basepos - 0.33
y = []
for i in x:
    y.append(
        (
            (
                statsRegTime[i]["totalMI" + IS_SHADOW]
                - statsRegTime[i]["gaussMI" + IS_SHADOW]
            )
            / (statsRegTime[i]["totalMI" + IS_SHADOW])
        )
    )  # +statsReg[i]["gaussMI"+IS_SHADOW]
plt.boxplot(
    y,
    positions=pos,
    widths=0.6,
    whis=2,
    boxprops={"color": "blue"},
    whiskerprops={"color": "blue"},
    flierprops={"color": "blue"},
    medianprops={"lw": 1.6, "color": "blue"},
    capprops={"color": "blue"},
)

pos = basepos + 0.33
y = []
for i in x:
    y.append(
        (
            (statsReg[i]["totalMI" + IS_SHADOW] - statsReg[i]["gaussMI" + IS_SHADOW])
            / (statsReg[i]["totalMI" + IS_SHADOW])
        )
    )  # +statsReg[i]["gaussMI"+IS_SHADOW]
plt.boxplot(
    y,
    positions=pos,
    widths=0.6,
    whis=2,
    boxprops={"color": "darkorange"},
    whiskerprops={"color": "darkorange"},
    flierprops={"color": "darkorange"},
    medianprops={"lw": 1.6, "color": "darkorange"},
    capprops={"color": "darkorange"},
)
p1 = plt.hlines(
    0.0, basepos[0] - 0.6, basepos[-1] + 0.6, "r", "solid", lw=1, label="%", zorder=1
)
p2 = plt.hlines(
    0.1, basepos[0] - 0.6, basepos[-1] + 0.6, "r", "dotted", lw=1, label="10%", zorder=1
)

plt.ylabel("relative difference in MI")
plt.xlabel("Band")
plt.xticks(
    ticks=basepos,
    labels=[
        r"$\delta$",
        r"$\theta$",
        r"$\alpha$",
        r"$\beta_{LOW}$",
        r"$\beta_{MID}$",
        r"$\beta_{HIGH}$",
        r"$\gamma$",
        r"$\beta_{ALL}$",
        r"$1-40 Hz$",
    ],
)
plt.legend(
    [
        Patch(facecolor="white", edgecolor="blue"),
        Patch(facecolor="white", edgecolor="darkorange"),
        p1,
        p2,
    ],
    ["Fixed time", "Fixed #samples", r"0", r"1%"],
)
plt.title("All subjects")
plt.show()
store_raw = np.array(y)  # [[0,1,2,7,6]]

In [ ]:
fig = plt.figure(figsize=(12, 6))
bpx = []
for i in x:
    bpx.append(statsReg[i]["totalMI"])
xpos = 2 * np.array(x) - 0.8
plt.boxplot(
    bpx,
    whis=2,
    positions=xpos,
    boxprops={"color": "b"},
    whiskerprops={"color": "b"},
    flierprops={"color": "b"},
    medianprops={"lw": 1.6, "color": "b"},
    capprops={"color": "b"},
)
bpx = []
for i in x:
    bpx.append(statsReg[i]["gaussMI"])
xpos = 2 * np.array(x) - 0.2
plt.boxplot(
    bpx,
    whis=2,
    positions=xpos,
    boxprops={"color": "darkorange"},
    whiskerprops={"color": "darkorange"},
    flierprops={"color": "darkorange"},
    medianprops={"lw": 1.6, "color": "darkorange"},
    capprops={"color": "darkorange"},
)
tkpos = 2 * np.arange(1, 10) - 0.5
plt.xticks(
    tkpos,
    labels=[
        r"$\delta$",
        r"$\theta$",
        r"$\alpha$",
        r"$\beta_{LOW}$",
        r"$\beta_{MID}$",
        r"$\beta_{HIGH}$",
        r"$\gamma$",
        r"$\beta_{ALL}$",
        r"$1-40 Hz$",
    ],
    rotation=45,
    fontsize="small",
)
plt.ylim(bottom=0)
plt.legend(
    [
        Patch(facecolor="white", edgecolor="b"),
        Patch(facecolor="white", edgecolor="darkorange"),
    ],
    ["Total MI", "Gauss MI"],
)
plt.title("Fixed #samples")
plt.xlabel("Band")
plt.ylabel("MI")
plt.show()

fig = plt.figure(figsize=(12, 6))
bpx = []
for i in x:
    bpx.append(statsRegTime[i]["totalMI"])
xpos = 2 * np.array(x) - 0.8
plt.boxplot(
    bpx,
    whis=2,
    positions=xpos,
    boxprops={"color": "b"},
    whiskerprops={"color": "b"},
    flierprops={"color": "b"},
    medianprops={"lw": 1.6, "color": "b"},
    capprops={"color": "b"},
)
bpx = []
for i in x:
    bpx.append(statsRegTime[i]["gaussMI"])
xpos = 2 * np.array(x) - 0.2
plt.boxplot(
    bpx,
    whis=2,
    positions=xpos,
    boxprops={"color": "darkorange"},
    whiskerprops={"color": "darkorange"},
    flierprops={"color": "darkorange"},
    medianprops={"lw": 1.6, "color": "darkorange"},
    capprops={"color": "darkorange"},
)
tkpos = 2 * np.arange(1, 10) - 0.5
plt.xticks(
    tkpos,
    labels=[
        r"$\delta$",
        r"$\theta$",
        r"$\alpha$",
        r"$\beta_{LOW}$",
        r"$\beta_{MID}$",
        r"$\beta_{HIGH}$",
        r"$\gamma$",
        r"$\beta_{ALL}$",
        r"$1-40 Hz$",
    ],
    rotation=45,
    fontsize="small",
)
plt.ylim(bottom=0)
plt.legend(
    [
        Patch(facecolor="white", edgecolor="b"),
        Patch(facecolor="white", edgecolor="darkorange"),
    ],
    ["Total MI", "Gauss MI"],
)
plt.title("Fixed time")
plt.xlabel("Band")
plt.ylabel("MI")
plt.show()

### All EEG

In [ ]:
STRETCH = 2
p1 = plt.hlines(0.0, 0.0, 9 * STRETCH, "r", "solid", lw=1, label="0", zorder=1)
p2 = plt.hlines(0.1, 0.0, 9 * STRETCH, "r", "dashed", lw=1, label="10%", zorder=1)
plt.boxplot(
    store_source.T,
    whis=2,
    positions=np.arange(9) * STRETCH + 1.5,
    widths=0.4,
    boxprops={"color": "darkorange"},
    whiskerprops={"color": "darkorange"},
    flierprops={"color": "darkorange"},
    medianprops={"lw": 1.6, "color": "darkorange"},
    capprops={"color": "darkorange"},
)
plt.boxplot(
    store_electrode.T,
    whis=2,
    positions=np.arange(9) * STRETCH + 1,
    widths=0.4,
    boxprops={"color": "blue"},
    whiskerprops={"color": "blue"},
    flierprops={"color": "blue"},
    medianprops={"lw": 1.6, "color": "blue"},
    capprops={"color": "blue"},
)
plt.boxplot(
    store_raw.T,
    whis=2,
    positions=np.arange(9) * STRETCH + 0.5,
    widths=0.4,
    boxprops={"color": "darkgreen"},
    whiskerprops={"color": "darkgreen"},
    flierprops={"color": "darkgreen"},
    medianprops={"lw": 1.6, "color": "darkgreen"},
    capprops={"color": "darkgreen"},
)
tkpos = np.arange(9) * STRETCH + 1
# plt.xticks(tkpos, x, rotation=45, fontsize="small")
plt.ylabel("Relative non-linearity in MI")
plt.xticks(
    ticks=np.arange(9) * STRETCH + 1,
    labels=[
        r"$\delta$",
        r"$\theta$",
        r"$\alpha$",
        r"$\beta_{LOW}$",
        r"$\beta_{MID}$",
        r"$\beta_{HIGH}$",
        r"$\gamma$",
        r"$\beta_{ALL}$",
        r"$1-40 Hz$",
    ],
)
plt.xlabel("Band")
plt.legend(
    [
        Patch(facecolor="white", edgecolor="darkgreen"),
        Patch(facecolor="white", edgecolor="blue"),
        Patch(facecolor="white", edgecolor="darkorange"),
        p1,
        p2,
    ],
    ["Electrode Voltage", "Electrode Power", "Source Power", r"0%", r"10%"],
    loc="center left",
    bbox_to_anchor=(1.0, 0.5),
    frameon=False,
)
plt.title("50 subjects $-$ rs-EEG $-$ Shadow")
plt.xlim((0.0, 9 * STRETCH))
plt.savefig("EEG_all_sha_New.pdf", bbox_inches="tight")
plt.ylim(top=0.27, bottom=-0.11)
plt.show()

## iEEG

In [ ]:
import glob
import os
import numpy as np
import matplotlib.pyplot as plt
import json
import re
from matplotlib.patches import Patch

statsReg = {}
for folder in glob.glob(
    "../../NonLinearityResultsNew/FIX_iEEG_fixedSamples_band?_bin10"
):
    if os.path.isfile(
        os.path.join(folder, os.path.split(folder)[1] + "_globalStats.json")
    ):
        try:
            statsMI = np.load(os.path.join(folder, f"subject00_10.npy"))
        except:
            statsMI = np.array([])
        pairs = statsMI.shape[0]
        band = int(re.findall(r"band(\d+)", folder)[0])
        print(folder, band, statsMI.shape)
        with open(
            os.path.join(folder, os.path.split(folder)[1] + "_globalStats.json")
        ) as fp:
            statsReg[band] = {k: np.array(v) for k, v in json.load(fp).items()}

statsRegTime = {}
for folder in glob.glob("../NonLinearityResultsNew/FIX_iEEG_fixedTime_band?_bin*"):
    if os.path.isfile(
        os.path.join(folder, os.path.split(folder)[1] + "_globalStats.json")
    ):
        try:
            assert False
            statsMI = np.load(os.path.join(folder, f"subject00_10.npy"))
        except:
            statsMI = np.array([])
        pairs = statsMI.shape[0]
        band = int(re.findall(r"band(\d+)", folder)[0])
        print(folder, band, statsMI.shape)
        with open(
            os.path.join(folder, os.path.split(folder)[1] + "_globalStats.json")
        ) as fp:
            statsRegTime[band] = {k: np.array(v) for k, v in json.load(fp).items()}

In [ ]:
IS_SHADOW = ""  # "shadow"#

In [ ]:
x = sorted(statsReg.keys())

all = np.zeros((19, len(x)))
for pat in range(19):
    y = [
        (
            (statsReg[i]["totalMI"][pat] - statsReg[i]["gaussMI"][pat])
            / (statsReg[i]["totalMI"][pat])
            if len(statsReg[i]["totalMI"]) > pat
            else np.NaN
        )
        for i in x
    ]  # 2*+statsReg[i]["gaussMI"][pat]
    plt.plot(x, y, "o--b", lw=1, alpha=0.5)
    all[pat, :] = y
my = np.nanmean(all, axis=0)
sy = np.nanstd(all, axis=0)
plt.plot([x[0], x[-1]], [0, 0], "-r", lw=1, label=0, zorder=1)
plt.fill_between(
    x,
    my - 2 * sy,
    my + 2 * sy,
    color="yellow",
    alpha=1,
    edgecolor=None,
    label=r"$\pm 2 \sigma$",
)
plt.fill_between(
    x,
    my - sy,
    my + sy,
    color="green",
    alpha=0.5,
    edgecolor=None,
    label=r"$\pm 1 \sigma$",
)
plt.plot(x, my, "--k", lw=2, label="mean")
plt.xticks(
    ticks=np.arange(1, 10),
    labels=[
        r"$\delta$",
        r"$\theta$",
        r"$\alpha$",
        r"$\beta_{LOW}$",
        r"$\beta_{MID}$",
        r"$\beta_{HIGH}$",
        r"$\gamma$",
        r"$\beta_{ALL}$",
        r"$1-40 Hz$",
    ],
)
plt.ylabel("relative difference in MI")
plt.xlabel("Band")
plt.legend()
plt.title("18 subjects $-$ iEEG")
plt.show()

In [ ]:
x = sorted(statsReg.keys())

y = []
for i in x:
    y.append(
        (
            (statsReg[i]["totalMI" + IS_SHADOW] - statsReg[i]["gaussMI" + IS_SHADOW])
            / (statsReg[i]["totalMI" + IS_SHADOW])
        )
    )  # 2*+statsReg[i]["gaussMI"+IS_SHADOW]
(p1,) = plt.plot([0.8, len(x) + 0.2], [0, 0], "-r", lw=1, label=0, zorder=1)
plt.boxplot(
    y,
    whis=2,
    boxprops={"color": "darkorange"},
    whiskerprops={"color": "darkorange"},
    flierprops={"color": "darkorange"},
    medianprops={"lw": 1.6, "color": "darkorange"},
    capprops={"color": "darkorange"},
)
tkpos = np.arange(len(x)) + 1
plt.xticks(tkpos, x, rotation=45, fontsize="small")
plt.ylabel("relative difference in MI")
plt.xlabel("iEEG")
# plt.xticks(ticks=np.arange(1,10), labels=[r"$\delta$",r"$\theta$",r"$\alpha$",r"$\beta_{LOW}$",r"$\beta_{MID}$",r"$\beta_{HIGH}$",r"$\gamma$",r"$\beta_{ALL}$",r"$1-40 Hz$"])
plt.legend(
    [Patch(facecolor="white", edgecolor="darkorange"), p1],
    ["Total MI \nvs Gauss MI", r"0"],
)
plt.title("All subjects")
plt.show()

STRETCH = 1.5
x = sorted(statsReg.keys())
basepos = np.arange(len(x)) * STRETCH
pos = basepos - 0.33
y = []
for i in x:
    y.append(
        (
            (
                statsRegTime[i]["totalMI" + IS_SHADOW]
                - statsRegTime[i]["gaussMI" + IS_SHADOW]
            )
            / (statsRegTime[i]["totalMI" + IS_SHADOW])
        )
    )  # +statsReg[i]["gaussMI"+IS_SHADOW]
plt.boxplot(
    y,
    positions=pos,
    widths=0.6,
    whis=2,
    boxprops={"color": "blue"},
    whiskerprops={"color": "blue"},
    flierprops={"color": "blue"},
    medianprops={"lw": 1.6, "color": "blue"},
    capprops={"color": "blue"},
)

pos = basepos + 0.33
y = []
for i in x:
    y.append(
        (
            (statsReg[i]["totalMI" + IS_SHADOW] - statsReg[i]["gaussMI" + IS_SHADOW])
            / (statsReg[i]["totalMI" + IS_SHADOW])
        )
    )  # +statsReg[i]["gaussMI"+IS_SHADOW]
plt.boxplot(
    y,
    positions=pos,
    widths=0.6,
    whis=2,
    boxprops={"color": "darkorange"},
    whiskerprops={"color": "darkorange"},
    flierprops={"color": "darkorange"},
    medianprops={"lw": 1.6, "color": "darkorange"},
    capprops={"color": "darkorange"},
)
p1 = plt.hlines(
    0.0, basepos[0] - 0.6, basepos[-1] + 0.6, "r", "solid", lw=1, label="%", zorder=1
)
p2 = plt.hlines(
    0.1, basepos[0] - 0.6, basepos[-1] + 0.6, "r", "dotted", lw=1, label="10%", zorder=1
)

plt.ylabel("relative difference in MI")
plt.xlabel("Band")
plt.xticks(
    ticks=basepos,
    labels=[
        r"$\delta$",
        r"$\theta$",
        r"$\alpha$",
        r"$\beta_{LOW}$",
        r"$\beta_{MID}$",
        r"$\beta_{HIGH}$",
        r"$\gamma$",
        r"$\beta_{ALL}$",
        r"$1-40 Hz$",
    ],
)
plt.legend(
    [
        Patch(facecolor="white", edgecolor="blue"),
        Patch(facecolor="white", edgecolor="darkorange"),
        p1,
        p2,
    ],
    ["Fixed time", "Fixed #samples", r"0", r"10%"],
)
plt.title("All subjects")
plt.show()

In [ ]:
x = sorted(statsReg.keys())

y = []
p1 = plt.hlines(0.0, 0.0, 5 + 1, "r", "solid", lw=1, label="0", zorder=1)
p2 = plt.hlines(0.1, 0.0, 5 + 1, "r", "dashed", lw=1, label="10%", zorder=1)
for i in [1, 2, 3, 8, 7]:
    y.append(
        ((statsReg[i]["totalMI"] - statsReg[i]["gaussMI"]) / (statsReg[i]["totalMI"]))
    )  # +statsReg[i]["gaussMI"]
plt.boxplot(
    y,
    whis=2,
    boxprops={"color": "darkorange"},
    whiskerprops={"color": "darkorange"},
    flierprops={"color": "darkorange"},
    medianprops={"lw": 1.6, "color": "darkorange"},
    capprops={"color": "darkorange"},
)
# tkpos=np.arange(len(x))+1
# plt.xticks(tkpos, x, rotation=45, fontsize="small")
plt.ylabel("Relative non-linearity in MI")
plt.xlabel("Band")
plt.xticks(
    ticks=np.arange(5) + 1,
    labels=[r"$\delta$", r"$\theta$", r"$\alpha$", r"$\beta$", r"$\gamma$"],
)
plt.legend(
    [Patch(facecolor="white", edgecolor="darkorange"), p1, p2],
    ["Total MI \nvs Gauss MI", r"0%", "10%"],
)
plt.title("18 subjects $-$ iEEG")
plt.xlim((0.0, 5 + 1))
plt.savefig("iEEG.pdf", bbox_inches="tight")
plt.show()

In [ ]:
x = sorted(statsReg.keys())

fig = plt.figure(figsize=(12, 6))
bpx = []
for i in x:
    bpx.append(statsReg[i]["totalMI"])
xpos = 2 * np.array(x) - 0.8
plt.boxplot(
    bpx,
    whis=2,
    positions=xpos,
    boxprops={"color": "b"},
    whiskerprops={"color": "b"},
    flierprops={"color": "b"},
    medianprops={"lw": 1.6, "color": "b"},
    capprops={"color": "b"},
)
bpx = []
for i in x:
    bpx.append(statsReg[i]["gaussMI"])
xpos = 2 * np.array(x) - 0.2
plt.boxplot(
    bpx,
    whis=2,
    positions=xpos,
    boxprops={"color": "darkorange"},
    whiskerprops={"color": "darkorange"},
    flierprops={"color": "darkorange"},
    medianprops={"lw": 1.6, "color": "darkorange"},
    capprops={"color": "darkorange"},
)
tkpos = 2 * np.arange(1, 10) - 0.5
plt.xticks(
    tkpos,
    labels=[
        r"$\delta$",
        r"$\theta$",
        r"$\alpha$",
        r"$\beta_{LOW}$",
        r"$\beta_{MID}$",
        r"$\beta_{HIGH}$",
        r"$\gamma$",
        r"$\beta_{ALL}$",
        r"$1-40 Hz$",
    ],
)
plt.ylim(bottom=0)
plt.legend(
    [
        Patch(facecolor="white", edgecolor="b"),
        Patch(facecolor="white", edgecolor="darkorange"),
    ],
    ["Total MI", "Gauss MI"],
)
plt.xlabel("iEEG")
plt.ylabel("MI")
plt.show()

## Spiking data

In [ ]:
import glob
import os
import numpy as np
import matplotlib.pyplot as plt
import json
import re
from matplotlib.patches import Patch

GOOD_SESSIONS = [
    831882777,
    816200189,
    771160300,
    786091066,
    779839471,
    778998620,
    781842082,
    778240327,
    793224716,
    839068429,
    794812542,
    768515987,
    767871931,
    840012044,
    766640955,
    847657808,
]

statsReg = [{} for __ in GOOD_SESSIONS]
for s, session in enumerate(GOOD_SESSIONS):
    for folder in glob.glob(f"../../NonLinearityResultsNew/spiking_{session}_*"):
        if os.path.isfile(
            os.path.join(folder, os.path.split(folder)[1] + "_globalStats.json")
        ):
            try:
                statsMI = np.load(os.path.join(folder, f"subject00_10.npy"))
            except:
                statsMI = np.array([])
            pairs = statsMI.shape[0]
            timeWindow = int(re.findall(r"_(\d{4})_", folder)[0])
            print(folder, timeWindow, statsMI.shape)
            with open(
                os.path.join(folder, os.path.split(folder)[1] + "_globalStats.json")
            ) as fp:
                statsReg[s][timeWindow] = {
                    k: np.array(v) for k, v in json.load(fp).items()
                }

In [ ]:
IS_SHADOW = ""  # "shadow"#
spikes = np.empty((len(statsReg[0]), len(statsReg[0][125]["totalMI"]), len(statsReg)))
ylabels = list(sorted(statsReg[0].keys()))
xlabels = [2**i for i in range(6)]
for s in range(len(GOOD_SESSIONS)):
    for t, timeWindow in enumerate(sorted(statsReg[0].keys())):
        spikes[t, :, s] = (
            statsReg[s][timeWindow]["totalMI" + IS_SHADOW]
            - statsReg[s][timeWindow]["gaussMI" + IS_SHADOW]
        ) / (statsReg[s][timeWindow]["totalMI" + IS_SHADOW])
this_session = 0
plt.imshow(spikes[:, :, this_session])
plt.yticks(np.arange(7), ylabels)
plt.xticks(np.arange(6), xlabels)
cbar = plt.colorbar()
cbar.ax.set_ylabel("RNL", rotation=90)
cbar.ax.get_yaxis().labelpad = 15
plt.xlabel("# units per site")
plt.ylabel("Time sample [ms]")
plt.title(GOOD_SESSIONS[this_session])
plt.show()
plt.imshow(np.mean(spikes, 2))
plt.yticks(np.arange(7), ylabels)
plt.xticks(np.arange(6), xlabels)
cbar = plt.colorbar()
cbar.ax.set_ylabel("RNL", rotation=90)
cbar.ax.get_yaxis().labelpad = 15
plt.xlabel("# units per site")
plt.ylabel("Time sample [ms]")
plt.title("Average")
plt.show()
# CA1      105
# VISp      92
# VISl      83
# VISrl     75
# VISam     71
# MGv       62
# ProS      53
# VISpm     50
# SUB       41
# VISal     32

In [ ]:
for i in range(len(ylabels)):
    plt.plot(
        np.mean(spikes, 2)[i, :],
        dashes=[i + 1, 1],
        color=(i / 6, 0, 1 - i / 6),
        label=ylabels[i],
    )
plt.legend(loc="center left", bbox_to_anchor=(1.0, 0.5), frameon=False, title="ms")
plt.xlabel("# units per site")
plt.ylabel("RNL")
plt.xticks(np.arange(6), xlabels)
plt.show()

In [ ]:
mean_spikes = np.mean(spikes, 2)
residual = np.empty_like(spikes)
for i in range(spikes.shape[2]):
    residual[:, :, i] = spikes[:, :, i] - mean_spikes
norms = np.linalg.norm(residual, ord="fro", axis=(0, 1))
closer_id = np.argmin(norms)
print(closer_id)
norms

In [ ]:
closer = GOOD_SESSIONS[closer_id]
plt.imshow(spikes[:, :, closer_id])
plt.yticks(np.arange(7), ylabels)
plt.xticks(np.arange(6), xlabels)
cbar = plt.colorbar()
cbar.ax.set_ylabel("RNL", rotation=90)
cbar.ax.get_yaxis().labelpad = 15
plt.xlabel("# units per site")
plt.ylabel("Time sample [ms]")
plt.title(closer)
plt.show()

In [ ]:
from mienc import Corrector
import seaborn as sns

sns.set_theme("talk", "ticks")

tru_maps = []
gau_maps = []
cor_maps = []
for folder in sorted(glob.glob(f"../../NonLinearityResultsNew/spiking_{closer}_*")):
    pairs = statsMI.shape[0]
    timeWindow = int(re.findall(r"_(\d{4})_", folder)[0])
    bins = int(re.findall(r"_bin(\d+)", folder)[0])
    with open(f"{folder}/shape.json") as fp:
        samples, groups, sizes = json.load(fp)
    c = Corrector(
        bins,
        samples,
        200,
        50000,
        samples,
        folder,
        "cache",
        1,
        False,
        False,
        True,
        "config.ini",
        True,
    )
    c.compute_correction()
    tru_maps.append(np.zeros((groups, groups, sizes)))
    gau_maps.append(np.zeros((groups, groups, sizes)))
    cor_maps.append(np.zeros((groups, groups, sizes)))
    for i in range(sizes):
        statsMI = np.load(os.path.join(folder, f"subject{i:02}_{bins}.npy"))
        statsMIcorrected = c.correct(statsMI)
        gaussMI = np.mean(statsMIcorrected[:, 1:], 1)
        tru_maps[-1][(*np.triu_indices(groups, 1), np.full(statsMI.shape[0], i))] = (
            statsMIcorrected[:, 0]
        )
        tru_maps[-1][:, :, i] += tru_maps[-1][:, :, i].T
        np.fill_diagonal(tru_maps[-1][:, :, i], np.inf)
        gau_maps[-1][(*np.triu_indices(groups, 1), np.full(statsMI.shape[0], i))] = (
            gaussMI[:]
        )
        gau_maps[-1][:, :, i] += gau_maps[-1][:, :, i].T
        np.fill_diagonal(gau_maps[-1][:, :, i], np.inf)
        correlation = np.load(os.path.join(folder, f"subject{i:02}_{bins}_cor.npy"))
        cor_maps[-1][
            (*np.triu_indices(groups, 1), np.full(statsMI.shape[0], i))
        ] = correlation
        cor_maps[-1][:, :, i] += cor_maps[-1][:, :, i].T + np.eye(
            groups
        )  # 0.9298734950321937*
    print(folder, timeWindow, statsMI.shape, samples, groups, sizes)
tru_maps = np.stack(tru_maps, -1)
gau_maps = np.stack(gau_maps, -1)
cor_maps = np.stack(cor_maps, -1)
mic_maps = -0.5 * np.log(1 - cor_maps**2)
rnl_maps = 1 - gau_maps / tru_maps

In [ ]:
minimum = np.min(rnl_maps[np.isfinite(rnl_maps)])
maximum = np.max(rnl_maps[np.isfinite(rnl_maps)])
# minimum = np.min(np.concatenate([a[np.isfinite(a)] for a in (tru_maps, gau_maps, mic_maps)]))
# maximum = np.max(np.concatenate([a[np.isfinite(a)] for a in (tru_maps, gau_maps, mic_maps)]))
np.sqrt(1 - np.exp(-2)), minimum, maximum

In [ ]:
group_size = 5  # 0 = 1 unit, ..., 5 = 32 units
time_width = 0  # 0 = 125 ms, ..., 6 = 8 s

cor_min = np.min(
    np.concatenate(
        [
            a[np.isfinite(a)]
            for a in (
                tru_maps[:, :, group_size, time_width],
                gau_maps[:, :, group_size, time_width],
                mic_maps[:, :, group_size, time_width],
            )
        ]
    )
)
cor_max = np.max(
    np.concatenate(
        [
            a[np.isfinite(a)]
            for a in (
                tru_maps[:, :, group_size, time_width],
                gau_maps[:, :, group_size, time_width],
                mic_maps[:, :, group_size, time_width],
            )
        ]
    )
)

fig, ax = plt.subplots(2, 3, gridspec_kw={"width_ratios": [6, 6, 1]})

plt.sca(ax[0, 0])
plt.imshow(
    tru_maps[:, :, group_size, time_width], vmin=cor_min, vmax=cor_max, cmap="viridis"
)
plt.title("True MI")
plt.xticks([])
plt.yticks([])
sns.despine(left=True, bottom=True)

plt.sca(ax[0, 1])
plt.imshow(
    gau_maps[:, :, group_size, time_width], vmin=cor_min, vmax=cor_max, cmap="viridis"
)
plt.title("Surrogates MI")
plt.xticks([])
plt.yticks([])
sns.despine(left=True, bottom=True)
plt.colorbar(ax=ax[0, 2], shrink=1.7, label="MI [nats]")

plt.sca(ax[1, 0])
plt.imshow(
    mic_maps[:, :, group_size, time_width], vmin=cor_min, vmax=cor_max, cmap="viridis"
)
plt.xlabel("Correlation MI")
plt.xticks([])
plt.yticks([])
sns.despine(left=True, bottom=True)

plt.sca(ax[1, 1])
plt.imshow(rnl_maps[:, :, group_size, time_width], vmin=-0.4, vmax=1)
plt.xlabel("RNL")
plt.xticks([])
plt.yticks([])
sns.despine(left=True, bottom=True)
plt.colorbar(ax=ax[1, 2], shrink=1.7, extend="min", extendfrac=0.03, label="RNL")


plt.sca(ax[0, 2])
plt.xticks([])
plt.yticks([])
sns.despine(left=True, bottom=True)
plt.sca(ax[1, 2])
plt.xticks([])
plt.yticks([])
sns.despine(left=True, bottom=True)
plt.subplots_adjust(hspace=0.01, wspace=-0.0)
plt.show()

In [ ]:
group_size = 5  # 0 = 1 unit, ..., 5 = 32 units
time_width = 0  # 0 = 125 ms, ..., 6 = 8 s

for group_size in range(5):
    for time_width in range(6):
        plt.scatter(
            gau_maps[:, :, group_size, time_width][np.triu_indices(groups, 1)],
            tru_maps[:, :, group_size, time_width][np.triu_indices(groups, 1)],
        )
plt.plot([0, 0.5], [0, 0.5], ":k")

# True - Shadow correlation

In [ ]:
import os
import json
import matplotlib.pyplot as plt
from matplotlib.colors import hsv_to_rgb
from matplotlib.lines import Line2D
import numpy as np


baseFolder = "../../NonLinearityResultsNew/"
all_RNL_true = {}
all_RNL_shadow = {}
s = 0.9

for folder in os.listdir(baseFolder):
    if "bin" in folder:
        name = "_".join(folder.split("_")[:-1])
        with open(os.path.join(baseFolder, folder, folder + "_globalStats.json")) as fp:
            tmp = {k: np.array(v) for k, v in json.load(fp).items()}
        all_RNL_true[name] = (tmp["totalMI"] - tmp["gaussMI"]) / tmp["totalMI"]
        all_RNL_shadow[name] = (tmp["totalMIshadow"] - tmp["gaussMIshadow"]) / tmp[
            "totalMIshadow"
        ]

datasets = [
    "shaved_eso245_cra_strin",
    "FIX_electrode_BLP",
    "FIX_source_BLP",
    "FIX_EEG_fixed",
    "FIX_iEEG_fixed",
]

handles = []
labels = []
plt.figure(figsize=(12, 8))
for i, dataset in enumerate(datasets):
    groupOne = sorted([k for k in all_RNL_true if dataset in k and "Time" not in k])
    groupTwo = sorted([k for k in all_RNL_true if dataset in k and "Time" in k])

    h = 0.2 * i
    vv = np.linspace(0.4, 0.9, len(groupOne))

    deltaH = 0.03 if len(groupTwo) else 0

    thisTrue = np.concatenate([all_RNL_true[k] for k in groupOne])
    thisShadow = np.concatenate([all_RNL_shadow[k] for k in groupOne])
    for j, k in enumerate(groupOne):
        color = hsv_to_rgb((h - deltaH, s, vv[j]))
        plt.scatter(all_RNL_true[k], all_RNL_shadow[k], s=2, color=color, alpha=0.5)
    plt.scatter(
        np.mean(thisTrue),
        np.mean(thisShadow),
        s=140,
        color=color,
        marker="*",
        zorder=10,
        edgecolors="w",
        linewidths=0.7,
    )
    plt.scatter(
        np.mean(thisTrue),
        np.mean(thisShadow),
        s=70,
        color=color,
        marker="*",
        zorder=10,
        edgecolors="k",
        linewidths=0.7,
    )
    label = " ".join([x for x in groupOne[0].split("_")[:-1] if x != "FIX"])
    handles.append(
        Line2D(
            [0],
            [0],
            marker="*",
            color="w",
            label=label,
            markerfacecolor=hsv_to_rgb((h - deltaH, s, 0.7)),
            markersize=10,
        )
    )
    labels.append(label + f" ({np.corrcoef(thisTrue, thisShadow)[0,1]:.3})")

    if len(groupTwo):
        thisTrue = np.concatenate([all_RNL_true[k] for k in groupTwo])
        thisShadow = np.concatenate([all_RNL_shadow[k] for k in groupTwo])
        for j, k in enumerate(groupTwo):
            color = hsv_to_rgb((h + deltaH, s, vv[j]))
            plt.scatter(all_RNL_true[k], all_RNL_shadow[k], s=2, color=color, alpha=0.5)
        label = " ".join([x for x in groupTwo[0].split("_")[:-1] if x != "FIX"])
        plt.scatter(
            np.mean(thisTrue),
            np.mean(thisShadow),
            s=140,
            color=color,
            marker="*",
            zorder=10,
            edgecolors="w",
            linewidths=0.7,
        )
        plt.scatter(
            np.mean(thisTrue),
            np.mean(thisShadow),
            s=70,
            color=color,
            marker="*",
            zorder=10,
            edgecolors="k",
            linewidths=0.7,
        )
        handles.append(
            Line2D(
                [0],
                [0],
                marker="*",
                color="w",
                label=label,
                markerfacecolor=hsv_to_rgb((h + deltaH, s, 0.7)),
                markersize=10,
            )
        )
        labels.append(label + f" ({np.corrcoef(thisTrue, thisShadow)[0,1]:.3})")

trueArr = np.concatenate([all_RNL_true[k] for k in all_RNL_true])
shadowArr = np.concatenate([all_RNL_shadow[k] for k in all_RNL_shadow])

plt.title(f"Correlation over sessions: {np.corrcoef(trueArr, shadowArr)[0,1]:.3}")

plt.legend(
    handles, labels, bbox_to_anchor=(0.95, 0.95), loc="upper right", frameon=False
)
plt.plot([-0.1, 0.1], [-0.1, 0.1], ":k", lw=0.8)
plt.xlabel("True RNL")
plt.ylabel("Shadow RNL")
plt.show()

In [ ]:
import os
import json
import matplotlib.pyplot as plt
from matplotlib.colors import hsv_to_rgb
from matplotlib.lines import Line2D
import numpy as np


baseFolder = "../../NonLinearityResultsNew/"
all_RNL_true = {}
all_RNL_shadow = {}
s = 0.9

for folder in os.listdir(baseFolder):
    if "bin" in folder:
        name = "_".join(folder.split("_")[:-1])
        with open(os.path.join(baseFolder, folder, folder + "_globalStats.json")) as fp:
            tmp = {k: np.array(v) for k, v in json.load(fp).items()}
        all_RNL_true[name] = (tmp["totalMI"] - tmp["gaussMI"]) / tmp["totalMI"]
        all_RNL_shadow[name] = (tmp["totalMIshadow"] - tmp["gaussMIshadow"]) / tmp[
            "totalMIshadow"
        ]

datasets = [
    "shaved_eso245_cra_strin",
    "FIX_electrode_BLP",
    "FIX_source_BLP",
    "FIX_EEG_fixed",
    "FIX_iEEG_fixed",
]

handles = []
labels = []
plt.figure(figsize=(12, 8))
for i, dataset in enumerate(datasets):
    groupOne = sorted([k for k in all_RNL_true if dataset in k and "Time" not in k])
    groupTwo = sorted([k for k in all_RNL_true if dataset in k and "Time" in k])

    h = 0.2 * i
    vv = np.linspace(0.5, 0.95, len(groupOne))

    deltaH = 0.03 if len(groupTwo) else 0

    thisTrue = np.array([np.mean(all_RNL_true[k]) for k in groupOne])
    thisShadow = np.array([np.mean(all_RNL_shadow[k]) for k in groupOne])
    for j, k in enumerate(groupOne):
        color = hsv_to_rgb((h - deltaH, s, vv[j]))
        plt.scatter(
            np.mean(all_RNL_true[k]),
            np.mean(all_RNL_shadow[k]),
            s=5,
            color=color,
            alpha=0.5,
        )
    plt.scatter(
        np.mean(thisTrue),
        np.mean(thisShadow),
        s=140,
        color=color,
        marker="*",
        zorder=10,
        edgecolors="w",
        linewidths=0.7,
    )
    plt.scatter(
        np.mean(thisTrue),
        np.mean(thisShadow),
        s=70,
        color=color,
        marker="*",
        zorder=10,
        edgecolors="k",
        linewidths=0.7,
    )
    label = " ".join([x for x in groupOne[0].split("_")[:-1] if x != "FIX"])
    handles.append(
        Line2D(
            [0],
            [0],
            marker="*",
            color="w",
            label=label,
            markerfacecolor=hsv_to_rgb((h - deltaH, s, 0.7)),
            markersize=10,
        )
    )
    labels.append(label + f" ({np.corrcoef(thisTrue, thisShadow)[0,1]:.3})")

    if len(groupTwo):
        thisTrue = np.array([np.mean(all_RNL_true[k]) for k in groupTwo])
        thisShadow = np.array([np.mean(all_RNL_shadow[k]) for k in groupTwo])
        for j, k in enumerate(groupTwo):
            color = hsv_to_rgb((h + deltaH, s, vv[j]))
            plt.scatter(
                np.mean(all_RNL_true[k]),
                np.mean(all_RNL_shadow[k]),
                s=5,
                color=color,
                alpha=0.5,
            )
        label = " ".join([x for x in groupTwo[0].split("_")[:-1] if x != "FIX"])
        plt.scatter(
            np.mean(thisTrue),
            np.mean(thisShadow),
            s=140,
            color=color,
            marker="*",
            zorder=10,
            edgecolors="w",
            linewidths=0.7,
        )
        plt.scatter(
            np.mean(thisTrue),
            np.mean(thisShadow),
            s=70,
            color=color,
            marker="*",
            zorder=10,
            edgecolors="k",
            linewidths=0.7,
        )
        handles.append(
            Line2D(
                [0],
                [0],
                marker="*",
                color="w",
                label=label,
                markerfacecolor=hsv_to_rgb((h + deltaH, s, 0.7)),
                markersize=10,
            )
        )
        labels.append(label + f" ({np.corrcoef(thisTrue, thisShadow)[0,1]:.3})")

trueArr = np.array([np.mean(all_RNL_true[k]) for k in all_RNL_true])
shadowArr = np.array([np.mean(all_RNL_shadow[k]) for k in all_RNL_shadow])

plt.title(
    f"Correlation of averages across subjects: {np.corrcoef(trueArr, shadowArr)[0,1]:.3}"
)

plt.legend(
    handles, labels, bbox_to_anchor=(0.98, 0.98), loc="upper right", frameon=False
)
plt.plot([0, 0.02], [0, 0.02], ":k", lw=0.8)
plt.xlabel("True RNL")
plt.ylabel("Shadow RNL")
plt.show()

In [ ]:
assert False

# Number of bins and number of samples effect

In [ ]:
import glob
import os
import numpy as np
import matplotlib.pyplot as plt
import json
import re
from matplotlib.patches import Patch

samples = {"many": {}, "sub": {}}
nctr = {"many": {}, "sub": {}}
for folder in glob.glob("../../NonLinearityResults/EEG_bands_band1_*"):
    if os.path.isfile(os.path.join(folder, "globalStats.json")):
        try:
            statfil = glob.glob(os.path.join(folder, f"patient00_*.npy"))
            statsMI = np.load(statfil[0])
        except:
            statsMI = np.array([])
        pairs = statsMI.shape[0]
        bins = int(re.findall(r"bin(\d+)", folder)[0])
        cat = "sub" if re.findall(r"sub", folder) else "many"
        print(folder, bins, statsMI.shape)
        with open(os.path.join(folder, "globalStats.json")) as fp:
            samples[cat][bins] = {k: np.array(v) for k, v in json.load(fp).items()}
        newco = np.load(os.path.join(folder, f"newco_{bins}.npy"))
        trueval = np.load(os.path.join(folder, f"trueval_{bins}.npy"))
        nctr[cat][bins] = np.stack([newco, trueval], -1)

In [ ]:
x = [8, 16, 50]
(p1,) = plt.plot([-1, 2 * len(x) - 2], [0, 0], "--r", lw=1, label=0, zorder=1)


y = []
for i in x:
    if i in samples["many"]:
        y.append(
            (
                (
                    samples["many"][i]["globaltotalMI"]
                    - samples["many"][i]["globalgaussMI"]
                )
                / (
                    samples["many"][i]["globaltotalMI"]
                    + samples["many"][i]["globalgaussMI"]
                )
            )[:41]
        )
    else:
        y.append([])
xpos = 2 * np.arange(len(x)) - 0.8
plt.boxplot(
    y,
    whis=2,
    positions=xpos,
    boxprops={"color": "darkorange"},
    whiskerprops={"color": "darkorange"},
    flierprops={"color": "darkorange"},
    medianprops={"lw": 1.6, "color": "darkorange"},
    capprops={"color": "darkorange"},
)


y = []
for i in x:
    if i in samples["sub"]:
        y.append(
            (
                (
                    samples["sub"][i]["globaltotalMI"]
                    - samples["sub"][i]["globalgaussMI"]
                )
                / (
                    samples["sub"][i]["globaltotalMI"]
                    + samples["sub"][i]["globalgaussMI"]
                )
            )[:41]
        )
    else:
        y.append([])
xpos = 2 * np.arange(len(x)) - 0.2
plt.boxplot(
    y,
    whis=2,
    positions=xpos,
    boxprops={"color": "b"},
    whiskerprops={"color": "b"},
    flierprops={"color": "b"},
    medianprops={"lw": 1.6, "color": "b"},
    capprops={"color": "b"},
)


tkpos = 2 * np.arange(len(x)) - 0.5
plt.xticks(tkpos, x, rotation=0, fontsize="small")
plt.ylabel("relative difference in MI")
plt.xlabel("# of bins")
plt.legend(
    [
        Patch(facecolor="white", edgecolor="darkorange"),
        Patch(facecolor="white", edgecolor="b"),
    ],
    ["100 Hz", "10 Hz"],
    title="Sampling",
    loc=2,
)
plt.title("All subjects")
plt.show()

In [ ]:
from matplotlib.lines import Line2D

plt.figure(figsize=(10, 8))
styles = {8: ":", 16: "--", 50: "-"}
colors = {"many": "darkorange", "sub": "b"}
names = {"many": "100 Hz", "sub": "10 Hz"}
for k in nctr:
    for k2 in nctr[k]:
        newco, trueval = nctr[k][k2].T
        plt.plot(trueval, newco, linestyle=styles[k2], color=colors[k])


plt.plot([min(trueval), max(trueval)], [min(trueval), max(trueval)], ":k")
handles = [Patch(facecolor="white", edgecolor="white")]
labels = ["Sampling"]
for k in colors:
    handles.append(Line2D([], [], linewidth=1.5, color=colors[k]))
    labels.append(names[k])
handles.append(Patch(facecolor="white", edgecolor="white"))
labels.append("Bins")
for k in styles:
    handles.append(Line2D([], [], linewidth=1.5, linestyle=styles[k], color="k"))
    labels.append(str(k))
plt.legend(handles, labels)
plt.xlim(left=0)
plt.show()

In [ ]:
import json
import numpy as np

with open(
    f"/home/raffaelli/NonLinearity/NonLinearityResults/100center_bin9/globalStats.json"
) as fp:
    gsa = {k: np.array(v) for k, v in json.load(fp).items()}

rnla = 1 - gsa["globalgaussMI"] / gsa["globaltotalMI"]
s_rnla = gsa["globalsigmaGaussMI"] / gsa["globaltotalMI"]
rnl_shaa = 1 - gsa["globalgaussMIshadow"] / gsa["globaltotalMIshadow"]
s_rnl_shaa = (
    gsa["globalsigmaGaussMIshadow"]
    * gsa["globalgaussMIshadow"]
    / gsa["globaltotalMIshadow"]
    * np.sqrt(1 / gsa["globalgaussMIshadow"] ** 2)
)  # + 99/gsa["globaltotalMIshadow"]**2)
mica = np.zeros(245)
for j in range(245):
    cor = np.load(
        f"/home/raffaelli/NonLinearityResults/100center_bin9/patient{j:02}_9_cor.npy"
    )
    mica[j] = np.mean(-0.5 * np.log(1 - cor**2))

In [ ]:
import matplotlib.pyplot as plt

ordera = np.argsort(rnla)
plt.figure(figsize=(8, 5))
xa = np.arange(rnla.shape[0])
p = plt.errorbar(xa, rnla[ordera], s_rnla[ordera], label="Real", elinewidth=1)
plt.errorbar(
    xa,
    rnl_shaa[ordera],
    s_rnl_shaa[ordera],
    label="Sha",
    elinewidth=1,
    ls=":",
    color=p.get_children()[0].get_color(),
)
plt.legend(loc="center left", bbox_to_anchor=(1.0, 0.5), frameon=False)
plt.xlabel("Time window")
plt.ylabel("RNL")
plt.title("Single voxel")
plt.show()

In [ ]:
plt.scatter(rnla[np.isfinite(mica)], rnl_shaa[np.isfinite(mica)], label="Raw")
plt.title(
    f"{np.corrcoef(rnla[np.isfinite(mica)], rnl_shaa[np.isfinite(mica)])[0,1]:.4}"
)
plt.xlabel(r"$RNL$")
plt.ylabel(r"$RNL_{SHA}$")
plt.legend()
plt.show()

plt.scatter(s_rnla[np.isfinite(mica)], 1 / mica[np.isfinite(mica)], label="Raw")
plt.title(
    r"$\rho=$"
    + f"{np.corrcoef(s_rnla[np.isfinite(mica)], 1/mica[np.isfinite(mica)])[0,1]:.4}"
)
plt.xlabel(r"$\sigma_{RNL}$")
plt.ylabel(r"$\frac{1}{\langle MI_{cor}\rangle}$")
plt.legend()
plt.show()

plt.scatter(1 / mica[np.isfinite(mica)], rnl_shaa[np.isfinite(mica)], label="Raw")
plt.title(
    r"$\rho=$"
    + f"{np.corrcoef(rnl_shaa[np.isfinite(mica)], 1/mica[np.isfinite(mica)])[0,1]:.4}"
)
plt.xlabel(r"$\frac{1}{\langle MI_{cor}\rangle}$")
plt.ylabel(r"$RNL_{SHA}$")
plt.legend()
plt.show()